# MichiganCast Demo 02 — Analysis and Baselines（EDA + 传统基线）

本 notebook 聚焦展示两类能力：

1. 数据分析能力（类别分布、季节性、相关性、漂移）。
2. 传统机器学习能力（统一口径下的 baseline 对比）。

## 0. 本章目标与结论

**目标：**
1. 使用 `src.analysis.eda_report` 生成自动分析报告。
2. 使用 `src.models.baselines` 生成传统模型基线结果。
3. 用统一指标口径解读模型能力边界。

**结论模板（执行后补充）：**
- 数据是否明显类别不平衡，是否需要 recall 优先策略。
- 传统模型中谁在 PR-AUC/F1/Recall 上表现最好。
- 多模态模型后续应重点改进的方向是什么。

## 1. 输入与输出（路径与产物）

| 类型 | 路径 |
|---|---|
| 清洗后输入数据 | `data/interim/traverse_city_daytime_clean_v1.csv` |
| EDA 总结报告 | `artifacts/reports/eda_summary.json` |
| EDA 图像目录 | `artifacts/figures/` |
| Baseline 结果 | `artifacts/reports/baseline_results.json` |
| Baseline 混淆矩阵图 | `artifacts/figures/baseline_cm_*.png` |

In [ ]:
from pathlib import Path
import subprocess
import json
import pandas as pd

CLEAN_CSV = 'data/interim/traverse_city_daytime_clean_v1.csv'
EDA_SUMMARY = 'artifacts/reports/eda_summary.json'
BASELINE_REPORT = 'artifacts/reports/baseline_results.json'
FIG_DIR = 'artifacts/figures'

print('clean csv exists:', Path(CLEAN_CSV).exists())
print('eda summary exists:', Path(EDA_SUMMARY).exists())
print('baseline report exists:', Path(BASELINE_REPORT).exists())

## 2. 方法与实现（调用 `src` 模块）

约定：默认只展示命令，防止误触发长时间训练；需要真正执行时把 `DEMO_EXECUTE=True`。

In [ ]:
DEMO_EXECUTE = False

def run_or_print(cmd: str):
    print('\n$ ' + cmd)
    if not DEMO_EXECUTE:
        print('[skip] DEMO_EXECUTE=False, only showing command.')
        return None
    proc = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(proc.stdout)
    if proc.returncode != 0:
        print(proc.stderr)
    return proc.returncode

### 2.1 生成 EDA 自动报告（T13）

**目的：** 用标准化脚本输出可复现分析结果，而不是手工画图。

In [ ]:
cmd_eda = (
    'scripts/run_in_pytorch_env.sh python -m src.analysis.eda_report '
    '--input data/interim/traverse_city_daytime_clean_v1.csv '
    '--summary artifacts/reports/eda_summary.json '
    '--fig-dir artifacts/figures'
)
run_or_print(cmd_eda)

### 2.2 训练传统基线模型（T14/T15）

**目的：** 在统一时间切分和指标口径下建立可比较基线。

In [ ]:
cmd_baseline = (
    'scripts/run_in_pytorch_env.sh python -m src.models.baselines '
    '--input data/interim/traverse_city_daytime_clean_v1.csv '
    '--report artifacts/reports/baseline_results.json '
    '--fig-dir artifacts/figures'
)
run_or_print(cmd_baseline)

## 3. 结果解读（图表与指标）

下面两个单元用于读取已有产物并快速形成演示表格。

In [ ]:
if Path(EDA_SUMMARY).exists():
    eda_payload = json.loads(Path(EDA_SUMMARY).read_text(encoding='utf-8'))
    class_summary = eda_payload.get('class_summary', {})
    print('rows_analyzed:', eda_payload.get('rows_analyzed'))
    print('positive_rate:', class_summary.get('positive_rate'))
    print('imbalance_conclusion:', class_summary.get('class_imbalance_conclusion'))
    print('\nEDA figures:')
    for k, v in eda_payload.get('figure_paths', {}).items():
        print(f'- {k}: {v}')
else:
    print('EDA summary not found. Run section 2.1 first.')

In [ ]:
if Path(BASELINE_REPORT).exists():
    baseline_payload = json.loads(Path(BASELINE_REPORT).read_text(encoding='utf-8'))
    rows = []
    for model_name, detail in baseline_payload.get('models', {}).items():
        val = detail.get('validation', {})
        rows.append({
            'model': model_name,
            'val_pr_auc': val.get('pr_auc'),
            'val_f1': val.get('f1'),
            'val_recall': val.get('recall'),
            'val_precision': val.get('precision'),
            'val_recall_at_precision': val.get('recall_at_precision'),
            'val_brier': val.get('brier'),
        })
    df_baseline = pd.DataFrame(rows).sort_values('val_pr_auc', ascending=False)
    display(df_baseline)
else:
    print('Baseline report not found. Run section 2.2 first.')

### 3.1 演示讲解要点（建议）

1. 为什么这里选 PR-AUC 而不是 Accuracy 作为主指标。
2. 类别不平衡对阈值策略有什么影响。
3. baseline 的上限意味着多模态模型需要在哪些方面超越。

## 4. 工程反思与下一步

- 反思：当前 baseline 是否已经充分（是否还应加 XGBoost/LightGBM）？
- 风险：如果数据分布漂移，baseline 指标可能出现什么变化？
- 下一步：进入 `03_multimodal_training_and_tuning.ipynb` 展示神经网络训练与调参闭环。

## Appendix. 复现实验命令

```bash
scripts/run_in_pytorch_env.sh python -m src.analysis.eda_report --input data/interim/traverse_city_daytime_clean_v1.csv --summary artifacts/reports/eda_summary.json --fig-dir artifacts/figures
scripts/run_in_pytorch_env.sh python -m src.models.baselines --input data/interim/traverse_city_daytime_clean_v1.csv --report artifacts/reports/baseline_results.json --fig-dir artifacts/figures
```